# 演習① SFT + LoRA（穴埋め形式）

`# TODO:` のコメント箇所を自分で実装してください。  
詰まったら `solutions/sol_01_sft.ipynb` を参照できます。

**目標**:
- LoRA 設定を自分で組み立てる
- SFTConfig の主要パラメータの意味を理解する
- rank を変えて学習可能パラメータ数の変化を確認する

In [ ]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, TaskType, get_peft_model
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset

os.environ.setdefault('HF_HOME', '/data/shared/hf_cache')
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'デバイス: {device}')

In [ ]:
# TODO: BitsAndBytesConfig を使って 4bit 量子化の設定を作成してください
# ヒント: load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype=torch.bfloat16

bnb_config = None  # TODO: BitsAndBytesConfig(...) で置き換える

MODEL_NAME = 'meta-llama/Meta-Llama-3-8B'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config if device == 'cuda' else None,
    device_map='auto' if device == 'cuda' else None,
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
print('モデル読み込み完了')

In [ ]:
# TODO: LoraConfig を設定してください
# 条件:
#   - task_type は CAUSAL_LM
#   - r（rank）は 16
#   - lora_alpha は 32
#   - target_modules に 'q_proj', 'k_proj', 'v_proj', 'o_proj' を含める

lora_config = None  # TODO: LoraConfig(...) で置き換える

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
# 期待される出力例: trainable params: 10,485,760 || all params: 8,040,767,488 || trainable%: 0.130

In [ ]:
# TODO: Alpaca 形式のプロンプト関数を完成させてください
# context がある場合は「### 文脈:\n{context}\n\n」を instruction と response の間に入れる

def format_instruction(sample: dict) -> str:
    instruction = sample.get('instruction', '')
    context     = sample.get('context', '')
    response    = sample.get('response', '')
    # TODO: context の有無で分岐し、適切なプロンプト文字列を返す
    return ''  # TODO: 実装する


# テスト
test_sample = {'instruction': 'Pythonとは？', 'context': '', 'response': 'プログラミング言語です。'}
result = format_instruction(test_sample)
print(result)
# 期待される出力:
# ### 指示:
# Pythonとは？
#
# ### 回答:
# プログラミング言語です。

In [ ]:
dataset = load_dataset('kunishou/databricks-dolly-15k-ja', split='train')
dataset = dataset.map(
    lambda x: {'text': format_instruction(x)},
    remove_columns=dataset.column_names,
)

# TODO: SFTConfig を設定してください
# 条件:
#   - output_dir='./outputs/ex01_sft'
#   - max_steps=20（デモ用）
#   - per_device_train_batch_size=4
#   - learning_rate=2e-4
#   - max_seq_length=512
#   - dataset_text_field='text'

training_args = None  # TODO: SFTConfig(...) で置き換える

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    tokenizer=tokenizer,
)

trainer.train()
print('学習完了！')

## 実験課題

1. `r=8`, `r=32`, `r=64` でそれぞれ `print_trainable_parameters()` を実行し、パラメータ数の変化を記録してください
2. 学習後に同じ質問に対して推論し、rank による出力の差を比較してください

解答は `solutions/sol_01_sft.ipynb` を参照してください。